# Use case (ARCO) — Sentinel-2 NDVI from a cloud-optimized Zarr store

This is the **memory-light** counterpart to `4.Use_case.ipynb`.

Instead of every student unzipping the SAFE archives and loading the
full-resolution JP2 bands into memory (which causes **out-of-memory** errors
when many run it at once), here we read a small, pre-built, chunked
**Analysis-Ready Cloud-Optimized (ARCO)** Zarr store straight from the **OSN**
object store. It opens lazily, needs **no credentials** (public bucket), and we
compute **NDVI at runtime** from the native bands.

The store was produced once by `source/build_arco.py`.

## Background & resources

New to **xarray** or **Zarr**? These give the background for this notebook:

- 📖 **xarray** — [Introducción a xarray](https://aladinor.github.io/AtmosCol-2023/introduccion-xarray/) (AtmosCol 2023): labelled N-dimensional arrays, `Dataset`/`DataArray`, indexing and lazy/Dask-backed reads.
- 📺 **Zarr v3** — [*From Tensors to Clouds: A Practical Guide to Zarr V3 and Zarr-Python 3*](https://www.youtube.com/watch?v=8FX4AXhRNMA) (PyData): chunked, compressed, cloud-native array storage — the format we read below.

**Zarr source.** This notebook points at a Zarr store on the **OSN** object store
(public, anonymous read):

```
s3://colombia-radar-arco/sentinel2-ard/T18NYM_20200205_20200210.zarr
endpoint: https://umn1.osn.mghpcc.org
```

It was produced once from the raw Sentinel-2 SAFE scenes by
`source/build_arco.py`; here we simply open it lazily and compute on it.

In [ ]:
import s3fs
import xarray as xr
import matplotlib as mpl
from dask_jobqueue import SLURMCluster

## 1. Set up a Dask cluster (distribute the computation)

We create the cluster **first**, so every operation below runs on the
distributed scheduler — open the *Dashboard* link to watch the chunks stream
through as we read from OSN and compute NDVI.

Use a `LocalCluster` here; on the HPC swap in the `SLURMCluster` (same config as
`4.Use_case.ipynb`) and scale out workers.

In [ ]:
cluster = SLURMCluster(cores=1, memory="8GB", processes=True,
                       scheduler_options={"dashboard_address": ":0"})
cluster.scale(jobs=2)

client = Client(cluster)
client

## 2. Open the ARCO store from OSN (anonymous)

The `colombia-radar-arco` bucket is public-read, so we open it with `anon=True`
— no AWS credentials required. The arrays come back lazy and Dask-backed, using
the same `100 x 100` chunking as the original notebook.

In [ ]:
fs = s3fs.S3FileSystem(
    anon=True,
    client_kwargs={"endpoint_url": "https://umn1.osn.mghpcc.org"},
)
store = s3fs.S3Map(
    "colombia-radar-arco/sentinel2-ard/T18NYM_20200205_20200210.zarr",
    s3=fs,
    check=False,
)

ds = xr.open_zarr(store, consolidated=False)  # lazy, dask-backed
ds

The dataset has the native 10 m bands stacked along `band`
(`b02, b03, b04, b08`, raw `uint16` digital numbers), a true-color `tci`
variable, and the two acquisitions stacked along `time`. Nothing has been
downloaded yet — the arrays are lazy Dask arrays.

## 3. Explore the individual bands (optional)

The same per-band views as `4.Use_case.ipynb` — **uncomment** to plot them
straight from the ARCO store (no unzip, no full-tile load). `isel(time=0)`
selects the first acquisition (2020-02-05).

In [ ]:
# True-color composite (TCI)
# ds.tci.isel(time=0).plot.imshow(figsize=(8, 7))

# Red band (B04)
# ds.reflectance.sel(band="b04").isel(time=0).plot(figsize=(8, 7))

# Near-infrared band (B08)
# ds.reflectance.sel(band="b08").isel(time=0).plot(figsize=(8, 7))

## 4. Compute NDVI at runtime

NDVI = (NIR − Red) / (NIR + Red) = (b08 − b04) / (b08 + b04). It is computed
lazily on the chunked store; because NDVI is a ratio it is invariant to the
common digital-number-to-reflectance scale, so the result matches the original
notebook exactly.

In [ ]:
refl = ds.reflectance.astype("float32")
nir = refl.sel(band="b08")
red = refl.sel(band="b04")

denom = nir + red
ndvi = ((nir - red) / denom).where(denom > 0)  # mask nodata (0/0)
ndvi

## 5. Trigger the distributed computation

Calling `.compute()` sends the per-chunk tasks to the cluster — watch the
Dashboard light up. The result for the first acquisition is pulled back as a
small in-memory array.

In [ ]:
result = ndvi.isel(time=0).compute()  # distributed over the 100x100 chunks
result

## 6. Plot NDVI with the same colormap as the original notebook

In [ ]:
cmap = mpl.colors.ListedColormap(
    [
        "#000000", "#a50026", "#d73027", "#f46d43",
        "#fdae61", "#fee08b", "#ffffbf", "#d9ef8b",
        "#a6d96a", "#66bd63", "#1a9850", "#006837",
    ]
)
bounds = [-1.0, -0.2, 0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
norm = mpl.colors.BoundaryNorm(bounds, cmap.N)

result.plot(cmap=cmap, norm=norm, figsize=(8, 7))

## 7. Bonus: NDVI change between the two dates

Because both scenes are time-stacked in one store on the same grid, we can
difference them directly — something the original single-scene notebook could
not do. This too runs on the cluster.

> **Caveat:** these two acquisitions are only **5 days apart**
> (2020-02-05 and 2020-02-10). Vegetation changes little over such a short
> interval, so this difference mainly highlights **clouds, water-level changes,
> and illumination/atmospheric differences** rather than real vegetation
> dynamics. It is shown here to demonstrate the *mechanics* of a time-series
> difference on an ARCO cube; for meaningful phenological change you would stack
> scenes weeks-to-months apart.

In [ ]:
ndvi_diff = (ndvi.isel(time=1) - ndvi.isel(time=0)).compute()
ndvi_diff.plot(cmap="RdBu", vmin=-0.5, vmax=0.5, figsize=(8, 7))

## 8. Release the cluster resources

Shut down the client and cluster when finished.

In [ ]:
client.close()
cluster.close()